<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi-e2/notebooks/gsm8k/Experiment_1_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


# Experiment 1 & 3: MEDAL vs. AB-MEDAL on GSM8K — Evaluation (20 Canonical + 50 Extended Questions)

This notebook runs **Experiment 1 (E1)** (Accuracy and Inference Efficiency) and **Experiment 3 (E3)** (Compute Allocation and Tree Topology Analysis) comparing **MEDAL** and **AB-MEDAL** on GSM8K with `LLaDA-8B-Instruct`.

---

### Dataset Split Organization (Aligned with E2 Evaluation):
1. **Canonical 20 Samples** (`sample_slice=(0, 20)`): The exact 20 frozen evaluation questions from E1 (`experiment_samples.json`), preserving historical prefix ordering.
2. **Additional 50 Samples** (`sample_slice=(20, 70)`): 50 extended validation questions sampled under the deterministic `sample_seed=1` order (`experiment_samples_e2.json`).
3. **Combined 70 Samples**: Full 70-question evaluation pool for aggregate metrics, paired McNemar comparisons, and offline E3 topology analysis.

---

## 1. Experimental Design & Controlled Isolation
The experiment isolates exactly one architectural difference:
- **MEDAL**: UCB selection policy with fixed sibling-oriented expansion (up to $K_2=5$ siblings per parent).
- **AB-MEDAL**: Adaptive Bayesian Student-t GEN/CONT selection with individual adaptive expansions.

| Component | MEDAL | AB-MEDAL |
|---|---|---|
| **Backbone Model** | LLaDA-8B-Instruct (bfloat16) | Same |
| **Questions** | Frozen GSM8K (20 Canonical + 50 Extended = 70 `evaluation`) | Same |
| **Prompt** | Two-shot decomposition (pens & muffins) | Same |
| **Search Budgets** | $B \in \{256, 512\}$ NFEs (Conditions A & B) | Same |
| **Max Search Depth** | $L_c = 10$ | Same |
| **Eligible Actions** | $K_1 = 3, K_2 = 5$ | Same |
| **Immediate Reward** | Normalized IG: $R_{IG}(p, s) = \frac{H(p) - H(s)}{H(p)}$ | Same |
| **Candidate Selection** | Minimum-$C=3$ rule: $d^* = \max\{d \in [1, L_c]: N_d \ge 3\}$ | Same |
| **Completion Decoder** | Deterministic Confidence at $T=0$ (256 length) | Same |
| **Completion Caching** | Shared partial states decode once | Same |
| **Search Tree Checkpointing** | Saved at exact budgets 256, 512 (Conditions A & B) | Same |

---

## 2. Continuous Search Checkpointing Optimization
Rather than executing two separate searches per question ($256 + 512 = 768$ NFEs), search runs continuously to $B=512$ (Condition B) and captures exact prefix snapshots at $B=256, 512$ (Conditions A & B).
Prefix equivalence ($T_{256}^{(512)} \equiv T_{256}^{(256)}$) ensures 100% bit-for-bit equivalence with independent runs while saving 256 NFEs per question.

---

## 3. Pure Offline E3 Analysis
E3 reads the saved E1 traces across the combined 70-question pool (or canonical 20 if run standalone) and generates comprehensive compute allocation metrics and all 6 publication figures **without performing any additional model forwards**.


## 1. Verify GPU Hardware
Verify that a CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM) is active.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.")
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / (1024 ** 3)
print(f"GPU: {gpu.name} | Total VRAM: {vram_gib:.1f} GiB")
if vram_gib < 20:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4 or A100 runtime with >= 22 GB VRAM; 16 GB T4 is too small.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bfloat16 inference.")
print("✓ GPU hardware verification passed.")


## 2. Repository Setup
Clone or update the GitHub repository on branch `adi-e2`. Supports private repository authentication via Colab Secrets or password prompt.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi-e2"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = "google.colab" in sys.modules or Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to private repo CooL-Legend/CSCI544-DLM: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local repository checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
for entry in (str(REPO_DIR), str(REPO_DIR / "src")):
    if entry not in sys.path:
        sys.path.insert(0, entry)


## 3. Install Dependencies
Install pinned LLaDA dependencies, editable package installation, and plotting tools.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "pandas", "seaborn"], check=True)
    for pkg in [*site.getsitepackages(), site.getusersitepackages()]:
        site.addsitedir(pkg)
print("✓ Dependencies installed and package paths loaded.")


## 4. Run Algorithmic Pre-Flight Unit Tests
Run unit tests with synthetic models to confirm checkpoint prefix equivalence, E1 dataset slicing, candidate selection, completion caching, and E3 offline metric accounting before loading weights.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/decoding/test_checkpoint_equivalence.py",
     "tests/tasks/test_experiment_1.py", "tests/tasks/test_e3_analysis.py", "-v"],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print("--- PRE-FLIGHT TESTS FAILED ---")
    print(res.stderr)
    raise RuntimeError("Algorithmic tests failed. Review environment before loading weights.")
print("✓ All pre-flight algorithmic unit tests passed.")


## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent caching of model weights and experimental artifacts across Colab sessions.

In [ ]:
from pathlib import Path
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
else:
    DRIVE_ROOT = REPO_DIR / ".cache/e1_eval"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")

OUTPUT_DIR = DRIVE_ROOT / "results/e1_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent Drive Root:", DRIVE_ROOT)
print("Experiment Output Directory:", OUTPUT_DIR)


## 6. Experiment Controls & Configuration
Configure the E2-aligned 70-question `evaluation` pool (20 canonical + 50 extended), budgets `[256, 512]` (Conditions A & B), depth `10`, minimum-`C=3` rule, and cooperative multi-question batch execution (`BATCH_SIZE = 4`).

In [ ]:
from dlm_project.tasks.gsm8k.experiment_1 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context
)

STAGE = "evaluation"
METHODS = ("medal", "ab_medal")
BUDGETS = tuple(CHECKPOINT_BUDGETS)  # (256, 512) (Conditions A & B)
MAX_DEPTH = MAX_SEARCH_DEPTH         # 10
MIN_C = MIN_CANDIDATES               # 3
# BATCH_SIZE = 4 is suited for A100 / High-RAM runtimes; use 2 if L4 VRAM is tight.
BATCH_SIZE = 4
RESUME = True
PROFILE_RESOURCES = False

base_ctx = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    sample_slice=(0, 70),
)

print(f"=== Experiment Configuration ({STAGE.upper()}) ===")
print("  Protocol:", base_ctx["protocol_version"])
print(f"  Stage: {base_ctx['stage']} ({len(base_ctx['examples'])} questions: 20 canonical + 50 extended)")
print("  Methods:", base_ctx["methods"])
print("  Budgets:", base_ctx["budgets"])
print("  Max Search Depth (Lc):", base_ctx["max_depth"])
print("  Candidate Selection Rule: min-C =", base_ctx["num_candidates"])
print("  Reward Mode:", base_ctx["reward_mode"])
print("  Batch Size:", BATCH_SIZE, "(cooperative varlen FlashAttention-2 batching)" if BATCH_SIZE > 1 else "(serial)")
print("  Resume:", RESUME, "| Profile Resources:", PROFILE_RESOURCES)


## 7. Cache & Load LLaDA-8B-Instruct
Stage model to fast local NVMe storage and load `LLaDA-8B-Instruct` model and tokenizer on CUDA in bfloat16.

In [ ]:
import shutil
from pathlib import Path
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_1 import load_model

profile = get_profile("llada")
model_name = profile["repo_id"].split("/")[-1]
drive_model_dir = DRIVE_ROOT / "models" / model_name
runtime_model_dir = Path("/content/models") / model_name

# Fast local NVMe cache in Colab runtime if available on Drive
if IN_COLAB and drive_model_dir.is_dir() and not (runtime_model_dir / "config.json").is_file():
    print(f"Copying model from Google Drive to local fast runtime storage: {runtime_model_dir}...")
    shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = str(runtime_model_dir if runtime_model_dir.is_dir() else (drive_model_dir if drive_model_dir.is_dir() else profile["repo_id"]))
print("Using model location:", MODEL_PATH)
print("Loading LLaDA-8B-Instruct wrapper on GPU in bfloat16...")
wrapper = load_model(base_ctx, model_path=MODEL_PATH, device="cuda", local_files_only=Path(MODEL_PATH).is_dir())
print(f"✓ Model loaded: {wrapper.identity['family']} ({wrapper.identity['repo_id']} @ {wrapper.identity['revision']})")


## 8. Execute Part 1: Canonical Evaluation Set (Original 20 Questions)
Evaluates the exact 20 evaluation questions from Experiment 1 (`sample_slice=(0, 20)`) with continuous prefix checkpointing, completion caching, and incremental Drive saving.

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_1 import run_e1_stage

ctx_20 = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    sample_slice=(0, 20),
)
ctx_20["setup"] = dict(base_ctx.get("setup", {}))
print(f"Starting Part 1 (Canonical 20 Questions) for {METHODS} at budgets {BUDGETS} (batch_size={BATCH_SIZE})...")
t0 = perf_counter()
q_rows_20, agg_rows_20 = run_e1_stage(
    wrapper,
    ctx_20,
    OUTPUT_DIR / "canonical_20",
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    resume=RESUME,
    batch_size=BATCH_SIZE,
    profile_resources=PROFILE_RESOURCES,
)
print(f"✓ Part 1 Complete in {(perf_counter() - t0) / 60:.2f} minutes ({len(q_rows_20)} checkpoint outcomes).")


## 9. Execute Part 2: Extended Validation Set (Additional 50 Questions)
Evaluates the additional 50 validation questions (`sample_slice=(20, 70)`) matching the E2 evaluation pool.

In [ ]:
ctx_50 = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    sample_slice=(20, 70),
)
ctx_50["setup"] = dict(base_ctx.get("setup", {}))
print(f"Starting Part 2 (Additional 50 Questions) for {METHODS} at budgets {BUDGETS} (batch_size={BATCH_SIZE})...")
t0 = perf_counter()
q_rows_50, agg_rows_50 = run_e1_stage(
    wrapper,
    ctx_50,
    OUTPUT_DIR / "extended_50",
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    resume=RESUME,
    batch_size=BATCH_SIZE,
    profile_resources=PROFILE_RESOURCES,
)
print(f"✓ Part 2 Complete in {(perf_counter() - t0) / 60:.2f} minutes ({len(q_rows_50)} checkpoint outcomes).")


## 10. Results & Paired Comparisons (Canonical 20, Extended 50, and Combined 70)
Display aggregate accuracy, Wilson 95% CIs, NFE utilization, simulation counts, wall-clock efficiency, and exact McNemar paired comparisons across Canonical 20, Extended 50, and Combined 70.

In [ ]:
import pandas as pd
from IPython.display import display
from dlm_project.experiments.sweep_report import write_csv
from dlm_project.tasks.gsm8k.experiment_1 import compute_e1_aggregate_rows
from scripts.run_e1_test import compute_paired_comparison

summaries_dir = OUTPUT_DIR / "summaries"
summaries_dir.mkdir(parents=True, exist_ok=True)

agg_cols = [
    "search_budget", "method", "sample_count", "accuracy_percent",
    "ci_95_low", "ci_95_high", "mean_nfe_utilization", "mean_search_nfe",
    "mean_simulations", "mean_completion_nfe", "mean_total_nfe", "mean_selected_depth",
    "mean_search_wall_seconds", "mean_completion_wall_seconds", "mean_total_wall_seconds", "fallback_rate",
]
paired_cols = [
    "search_budget", "paired_questions", "medal_accuracy", "ab_medal_accuracy",
    "accuracy_delta_percent", "ab_medal_only_correct", "medal_only_correct", "mcnemar_exact_p",
]

combined_q_rows = q_rows_20 + q_rows_50
combined_agg_rows = compute_e1_aggregate_rows(combined_q_rows, STAGE, METHODS, BUDGETS)
if combined_q_rows:
    write_csv(summaries_dir / "e1_combined_70_per_question.csv", combined_q_rows)
if combined_agg_rows:
    write_csv(summaries_dir / "e1_combined_70_aggregate.csv", combined_agg_rows)

for title, rows, agg, slug in [
    ("Part 1: Canonical 20 Questions", q_rows_20, agg_rows_20, "canonical_20"),
    ("Part 2: Additional 50 Questions", q_rows_50, agg_rows_50, "extended_50"),
    ("Combined Pool (All 70 Questions)", combined_q_rows, combined_agg_rows, "combined_70"),
]:
    if agg:
        df_agg = pd.DataFrame(agg)
        print(f"=== {title} Aggregate Metrics ===")
        display(df_agg[[c for c in agg_cols if c in df_agg.columns]])
    paired = compute_paired_comparison(rows, budgets=BUDGETS)
    if paired:
        df_paired = pd.DataFrame(paired)
        print(f"=== {title} Paired Comparison (AB-MEDAL vs. MEDAL) ===")
        display(df_paired[[c for c in paired_cols if c in df_paired.columns]])
        write_csv(summaries_dir / f"e1_{slug}_paired_comparison.csv", paired)
        if slug == "combined_70":
            write_csv(summaries_dir / "e1_evaluation_paired_comparison.csv", paired)

if combined_q_rows:
    df_pq = pd.DataFrame(combined_q_rows)
    pq_cols = [
        "example_id", "search_budget", "method", "exact_match",
        "predicted_answer", "reference_answer", "selected_depth",
        "search_nfes", "completion_nfes", "total_nfes",
        "nfe_utilization", "simulations_completed", "was_cached",
    ]
    print("\n=== Sample Per-Question Outcomes (First 20 Rows) ===")
    display(df_pq[[c for c in pq_cols if c in df_pq.columns]].head(20))


## 11. Interactive Search Tree & Decision Trace Inspector
Examine candidate shortlists, NFE utilization, selection gap $\Delta_d = d_{\max} - d^*$, and Student-t GEN/CONT posterior traces for any evaluated question.

In [ ]:
import json
from pathlib import Path

INSPECT_SPLIT = "canonical_20"  # "canonical_20" or "extended_50"
INSPECT_QUESTION_IDX = 0
INSPECT_BUDGET = 256

split_examples = (ctx_20 if INSPECT_SPLIT == "canonical_20" else ctx_50)["examples"]
q_info = split_examples[INSPECT_QUESTION_IDX]
ex_id = q_info["example_id"]
split_stage_dir = OUTPUT_DIR / INSPECT_SPLIT / STAGE

print(f"=== Inspecting {INSPECT_SPLIT} Question {ex_id} (Budget {INSPECT_BUDGET}) ===")
print("Question:\n", q_info["question"])
print("Reference Answer:", q_info.get("reference_solution", "").split("####")[-1].strip())
print(f"Results Directory: {split_stage_dir} (exists: {split_stage_dir.exists()})")

for m in METHODS:
    chk_dir = split_stage_dir / m / ex_id / "checkpoints" / str(INSPECT_BUDGET)
    sel_path = chk_dir / "selection.json"
    met_path = chk_dir / "metrics.json"
    tree_path = chk_dir / "tree.json"

    if sel_path.is_file() or met_path.is_file():
        sel_record = json.loads(sel_path.read_text()) if sel_path.is_file() else {}
        met_record = json.loads(met_path.read_text()) if met_path.is_file() else {}

        selected_id = sel_record.get("selected_node_id", met_record.get("selected_node_id"))
        selected_depth = sel_record.get("selected_depth", met_record.get("selected_depth"))
        selected_q = met_record.get("selected_q", sel_record.get("selected_q"))
        max_depth = met_record.get("max_depth_reached", sel_record.get("max_depth_reached", "N/A"))
        gap = met_record.get("selection_depth_gap", sel_record.get("selection_depth_gap", "N/A"))
        distinct_by_depth = met_record.get("distinct_candidates_by_depth") or sel_record.get("distinct_candidates_by_depth")
        shortlist_ids = sel_record.get("shortlist_ids") or met_record.get("shortlist_node_ids", [])

        print(f"\n--- {m.upper()} Checkpoint Metrics & Selection ({ex_id} @ B={INSPECT_BUDGET}) ---")
        print(f"  Selected Node: {selected_id} | Depth: {selected_depth} | Q: {selected_q:.4f}" if selected_q is not None else f"  Selected Node: {selected_id} | Depth: {selected_depth}")
        print(f"  Exact Match: {met_record.get('exact_match')} | Pred: {met_record.get('predicted_answer')} | Ref: {met_record.get('reference_answer')}")
        print(f"  NFE Utilization: {met_record.get('nfe_utilization', 1.0) * 100:.1f}% | Sims: {met_record.get('simulations_completed', 'N/A')} | NFEs (search/comp/total): {met_record.get('search_nfes')}/{met_record.get('completion_nfes')}/{met_record.get('total_nfes')}")
        print(f"  Max Depth Reached: {max_depth} | Selection Gap (Δd): {gap}")
        print(f"  Distinct Candidates by Depth: {distinct_by_depth}")
        print(f"  Shortlist Node IDs: {shortlist_ids}")
    else:
        print(f"\n--- {m.upper()} Selection Record ---")
        print(f"  [Not Found] Expected path: {chk_dir}")

dec_path = split_stage_dir / "ab_medal" / ex_id / "decisions.jsonl"
if dec_path.is_file():
    dec_events = [json.loads(line) for line in dec_path.read_text().splitlines() if line.strip()]
    print(f"\n--- AB-MEDAL Decision Trace Sample (first 10 of {len(dec_events)} decisions) ---")
    for d in dec_events[:10]:
        print(f"  Cycle {d.get('cycle', d.get('iteration'))}: parent node {d.get('node_id')} -> {d.get('choice', d.get('decision_type'))} (forced={d.get('forced')})")


## 12. E3 Offline Metric Extraction & Publication Figures
Run offline tree allocation, branching dynamics, and structural analysis on saved evaluation checkpoints across the combined 70-question pool (zero GPU compute), and generate 6 publication figures.

In [ ]:
import shutil
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

combined_results_dir = OUTPUT_DIR / "combined_70" / STAGE
combined_results_dir.mkdir(parents=True, exist_ok=True)
for part_dir in [OUTPUT_DIR / "canonical_20" / STAGE, OUTPUT_DIR / "extended_50" / STAGE]:
    if part_dir.is_dir():
        for m in METHODS:
            m_src = part_dir / m
            if m_src.is_dir():
                m_dst = combined_results_dir / m
                m_dst.mkdir(parents=True, exist_ok=True)
                for q_dir in m_src.iterdir():
                    if q_dir.is_dir() and not (m_dst / q_dir.name).exists():
                        try:
                            (m_dst / q_dir.name).symlink_to(q_dir.resolve(), target_is_directory=True)
                        except OSError:
                            shutil.copytree(q_dir, m_dst / q_dir.name)

has_combined = any((combined_results_dir / m).is_dir() and any((combined_results_dir / m).iterdir()) for m in METHODS)
eval_results_dir = combined_results_dir if has_combined else (OUTPUT_DIR / "canonical_20" / STAGE)
e3_output_dir = OUTPUT_DIR / ("combined_70" if has_combined else "canonical_20") / "summaries"
e3_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Running E3 Offline Tree Analysis on {eval_results_dir}...")
e3_q, e3_agg = analyze_e3_stage(
    results_dir=eval_results_dir,
    output_dir=e3_output_dir,
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print(f"✓ E3 Analysis Completed successfully: {len(e3_q)} question outcomes evaluated.")

figures_dir = e3_output_dir / "e3_figures"
fig_titles = {
    "fig1_nfe_by_depth": "Figure 1: NFE Allocation by Tree Depth — MEDAL vs AB-MEDAL across budgets",
    "fig2_branching_factors": "Figure 2: Branching Factor Distribution — Children per expanded parent",
    "fig3_candidate_density": "Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth",
    "fig4_root_subtrees": "Figure 4: Root-Subtree Compute Concentration — Search effort across root branches",
    "fig5_reward_redundancy": "Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states",
    "fig6_accuracy_vs_budget": "Figure 6: Accuracy vs Search Budget — Performance scaling curve",
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f"{fig_key}.png"
    if png_path.is_file():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(png_path), width=750))


## 13. Synchronize All Artifacts to Google Drive
Explicitly flush and sync all in-memory filesystem buffers to Google Drive cloud storage so all traces, trees, completions, CSVs, and plots are safely persisted and reconstructible even if Colab disconnects.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount("/content/drive")
    print("✓ Google Drive synchronization complete. All artifacts safely flushed to Drive.")
else:
    print(f"✓ Local artifacts safely saved at: {OUTPUT_DIR}")
